# I-JEPA single-GPU — 5-epoch run, then resume on epoch 6

End-to-end acceptance test for the **single-GPU / multi-process** refactor of
`ijepa-main-single-GPU`. Every step records a `PASS`/`FAIL` into a check registry; the last
cell prints one verdict table.

| # | Step |
|---|---|
| 1 | Probe the runtime, require a GPU |
| 2 | Upload the repo as `.zip`, unzip it, **fingerprint the source** to confirm the fixes are present |
| 3 | Copy the DR subset `.zip` from Drive → local disk, unzip **locally**, validate the images |
| 4–5 | Write configs, build the launcher |
| 6 | **Run A** — 5 epochs from scratch |
| 7 | Inspect the checkpoint + round-trip `_match_state_dict_prefix` |
| 8–9 | **Run B** — resume, run the 6th epoch only, verify it truly resumed |
| 10 | Results: per-epoch table + loss curve |
| 11 | **Verdict** |

## What is under test

| Change | Location | How this notebook proves it |
|---|---|---|
| `init_distributed` early-returns `(1,0)`, no NCCL init | `src/utils/distributed.py:39` | Run A reaches epoch 1 instead of hanging |
| DDP wrap gated on `world_size > 1` | `src/train.py:222` | Saved `state_dict` keys carry **no** `module.` prefix |
| `_DummySampler` replaces `DistributedSampler` | `src/datasets/imagenet1k.py:56` | Dataloader yields batches, `ipe > 0` |
| `main.py` runs inline for 1 GPU, no `mp.Process` | `main.py:72` | Child stdout streams into this notebook |
| `_match_state_dict_prefix` normalises `module.` | `src/helper.py:23` | Cell 7 round-trips a synthetic DDP checkpoint |
| Resume path | `src/train.py:236-249` | Run B loads epoch 5 and executes epoch 6 **only** |

---

### Before you start

**Runtime → Change runtime type → GPU.** Then zip the repo on your Windows machine:

```powershell
Compress-Archive -Path "C:\Users\khali\Projects\2026 retina deseases\ijepa-main-single-GPU\*" `
                 -DestinationPath "$env:USERPROFILE\Downloads\ijepa-main-single-GPU.zip" -Force
```

Run every cell in order, top to bottom.

## 1 · Runtime probe

Fails fast if there is no GPU — the rest of the notebook would waste a Colab session on CPU.
Also sets up the check registry every later cell writes into.

In [ ]:
#@title 1 · Runtime probe + check registry
import os, sys, platform, shutil, textwrap

# ---- check registry: every assertion in this notebook lands here -----------------
CHECKS = []

def check(name, ok, detail=""):
    # record a soft check; returns the boolean so callers can branch
    ok = bool(ok)
    CHECKS.append({"name": name, "ok": ok, "detail": str(detail)})
    print(f"  [{'PASS' if ok else 'FAIL'}] {name}" + (f"  --  {detail}" if detail else ""))
    return ok

def require(name, ok, detail=""):
    # record a check and stop the cell if it failed
    if not check(name, ok, detail):
        raise AssertionError(f"{name}: {detail}")
    return True

def checks_reset(prefix):
    # drop earlier results for a prefix so re-running a cell doesn't double-count
    global CHECKS
    CHECKS = [c for c in CHECKS if not c["name"].startswith(prefix)]

# ---- probe ----------------------------------------------------------------------
import torch, torchvision, numpy, yaml, PIL, matplotlib

print(f"python        : {sys.version.split()[0]}   ({platform.platform()})")
print(f"torch         : {torch.__version__}")
print(f"torchvision   : {torchvision.__version__}")
print(f"numpy / yaml  : {numpy.__version__} / {yaml.__version__}")
print(f"pillow / mpl  : {PIL.__version__} / {matplotlib.__version__}")
# the repo imports only the above; submitit is needed by main_distributed.py alone,
# which this notebook never runs -- so there is nothing to pip install.

HAS_GPU = torch.cuda.is_available()
if HAS_GPU:
    p = torch.cuda.get_device_properties(0)
    GPU_NAME, GPU_MEM = p.name, p.total_memory / 1024**3
    print(f"\ngpu           : {GPU_NAME}  ({GPU_MEM:.1f} GiB, sm_{p.major}{p.minor})")
    print(f"bf16 supported: {torch.cuda.is_bf16_supported()}")
else:
    GPU_NAME, GPU_MEM = "none", 0.0
    print("\ngpu           : NONE")

total, _, free = shutil.disk_usage("/content")
print(f"disk /content : {free/1024**3:.1f} GiB free of {total/1024**3:.1f} GiB\n")

require("env: GPU available", HAS_GPU,
        "Runtime > Change runtime type > T4/A100, then re-run from cell 1")
check("env: >=15 GiB free disk", free / 1024**3 >= 15, f"{free/1024**3:.1f} GiB free")

## 2 · Upload the repo zip, unzip, and fingerprint the source

The upload is the default path; set `REPO_ZIP` to a Drive path to skip the prompt.

**The fingerprint matters more than the unzip.** If you upload a stale zip — or pristine
upstream `facebookresearch/ijepa` — every later cell still *runs*, and the resume test still
*passes* for the wrong reasons. So this cell greps the extracted source for each of the five
changes and refuses to continue if `_match_state_dict_prefix` is missing.

In [ ]:
#@title 2 · Upload repo zip -> unzip -> fingerprint the source
import zipfile, shutil, os, re
from pathlib import Path

REPO_ZIP = None   #@param {type:"raw"}
# ^ leave None to get an upload prompt, or set e.g.
#   "/content/drive/MyDrive/ijepa-main-single-GPU.zip"

UNPACK = Path("/content/repo_unpack")
checks_reset("repo:")

if REPO_ZIP is None:
    from google.colab import files
    up = files.upload()                                # <-- pick ijepa-main-single-GPU.zip
    require("repo: a file was uploaded", len(up) == 1,
            f"expected exactly 1 zip, got {list(up)}")
    name = next(iter(up))
    REPO_ZIP = Path("/content") / name
    REPO_ZIP.write_bytes(up[name])                     # don't rely on files.upload's cwd
REPO_ZIP = Path(REPO_ZIP)
require("repo: zip exists", REPO_ZIP.exists(), str(REPO_ZIP))
require("repo: zip is a zip", zipfile.is_zipfile(REPO_ZIP), str(REPO_ZIP))
print(f"\nzip: {REPO_ZIP}  ({REPO_ZIP.stat().st_size/1024**2:.2f} MiB)")

shutil.rmtree(UNPACK, ignore_errors=True); UNPACK.mkdir(parents=True)
with zipfile.ZipFile(REPO_ZIP) as z:
    z.extractall(UNPACK)

# the zip may or may not wrap everything in a top-level folder -- locate the dir that
# actually holds main.py + src/train.py rather than assuming either layout
def is_repo_root(d: Path) -> bool:
    return (d / "main.py").is_file() and (d / "src" / "train.py").is_file()

REPO = next((d for d in [UNPACK, *UNPACK.rglob("*")] if d.is_dir() and is_repo_root(d)), None)
require("repo: root located", REPO is not None,
        "no dir containing both main.py and src/train.py found inside the zip")
for junk in list(REPO.rglob("__pycache__")):           # strip anything that rode along
    shutil.rmtree(junk, ignore_errors=True)
print(f"repo root: {REPO}\n")

# ---- fingerprint: are the single-GPU changes actually in this copy? --------------
FINGERPRINT = [
    ("src/helper.py",             r"def _match_state_dict_prefix",
     "checkpoint module.-prefix normalisation", True),
    ("src/train.py",              r"if world_size > 1:",
     "DDP wrap gated on world_size", True),
    ("src/utils/distributed.py",  r"if int\(world_size\) <= 1:",
     "init_distributed skips NCCL for 1 rank", True),
    ("src/datasets/imagenet1k.py", r"_DummySampler",
     "non-distributed sampler fallback", True),
    ("main.py",                   r"if num_gpus > 1:",
     "inline run for a single device", False),
]
print("source fingerprint:")
for rel, pat, what, hard in FINGERPRINT:
    f = REPO / rel
    found = f.is_file() and re.search(pat, f.read_text(encoding="utf-8")) is not None
    (require if hard else check)(f"repo: {what}", found, rel if found else f"NOT FOUND in {rel}")

print("\nThis zip is the refactored single-GPU tree, not pristine upstream.\n")
for f in sorted(REPO.iterdir()):
    print(f"  {'d' if f.is_dir() else '-'} {f.name}")

## 3 · Copy the subset zip from Drive, unzip it **locally**

`shutil.copy2` pulls the zip onto the runtime's own disk first; extraction happens there.
Reading an `ImageFolder` straight off the Drive FUSE mount makes per-image latency the
bottleneck, so the local copy is the point of this cell, not an optimisation to skip.

Default is Track A's `subset_frac10_strat_noqa.zip` (3512 images, 5 grade classes). Any zip
whose internal layout contains `<class>/` folders of images works — the cell locates the
`ImageFolder` root itself, then opens one image per class to confirm the decoder is happy
before an epoch's worth of time is spent finding out.

In [ ]:
#@title 3 · Drive subset zip -> local copy -> local unzip -> validate
import zipfile, shutil, os, time
from pathlib import Path
from collections import Counter
from PIL import Image

DRIVE_SUBSET_ZIP = "/content/drive/MyDrive/retina_phase1/subsets/subset_frac10_strat_noqa.zip"  #@param {type:"string"}
DATA_ROOT    = Path("/content/data/ijepa")   # -> config data.root_path
IMAGE_FOLDER = "dr_subset"                   # -> config data.image_folder
LOCAL_ZIP    = Path("/content/subset_local.zip")
RAW          = Path("/content/data/_raw")
checks_reset("data:")

from google.colab import drive
drive.mount("/content/drive")
src = Path(DRIVE_SUBSET_ZIP)
require("data: subset zip on Drive", src.exists(), str(src))

# skip the (slow) Drive read if an identical local copy is already here
if LOCAL_ZIP.exists() and LOCAL_ZIP.stat().st_size == src.stat().st_size:
    print(f"local copy already present, size matches -> reusing {LOCAL_ZIP}")
else:
    t0 = time.time()
    shutil.copy2(src, LOCAL_ZIP)                       # Drive -> local disk
    print(f"copied {src.name} -> {LOCAL_ZIP}  "
          f"({LOCAL_ZIP.stat().st_size/1024**2:.1f} MiB in {time.time()-t0:.1f}s)")
require("data: zip copied locally", zipfile.is_zipfile(LOCAL_ZIP), str(LOCAL_ZIP))

shutil.rmtree(RAW, ignore_errors=True); RAW.mkdir(parents=True)
t0 = time.time()
with zipfile.ZipFile(LOCAL_ZIP) as z:
    n_members = len(z.namelist())
    z.extractall(RAW)                                  # extracted on local disk
print(f"unzipped {n_members} members locally in {time.time()-t0:.1f}s")

IMG_EXT = {".jpg", ".jpeg", ".png", ".ppm", ".bmp", ".tif", ".tiff", ".webp"}
def is_imagefolder_root(d: Path) -> bool:
    subs = [s for s in d.iterdir() if s.is_dir()]
    return len(subs) >= 2 and any(
        any(f.suffix.lower() in IMG_EXT for f in s.iterdir() if f.is_file()) for s in subs)

root = next((d for d in [RAW, *RAW.rglob("*")] if d.is_dir() and is_imagefolder_root(d)), None)
require("data: ImageFolder root found", root is not None,
        "no dir with >=2 class subdirs of images inside the zip")

# src/datasets/imagenet1k.py:117 appends a 'train/' suffix -> expose the class dirs there
train_link = DATA_ROOT / IMAGE_FOLDER / "train"
train_link.parent.mkdir(parents=True, exist_ok=True)
if train_link.exists() or train_link.is_symlink():
    train_link.unlink()
os.symlink(root, train_link)                           # local -> local, no second copy

counts, samples = Counter(), []
for c in sorted(p for p in train_link.iterdir() if p.is_dir()):
    imgs = [f for f in c.rglob("*") if f.suffix.lower() in IMG_EXT]
    counts[c.name] = len(imgs)
    if imgs:
        samples.append(imgs[0])
N_IMAGES  = sum(counts.values())
N_CLASSES = len(counts)

print(f"\nImageFolder root : {root}")
print(f"exposed as       : {train_link}")
print(f"classes ({N_CLASSES})      : {dict(counts)}")
print(f"images           : {N_IMAGES}\n")

require("data: images extracted", N_IMAGES > 0, f"{N_IMAGES} images")
check("data: >=2 classes", N_CLASSES >= 2, f"{N_CLASSES} class dirs")

# open one image per class -- a corrupt/unsupported decode should surface now, not
# three minutes into epoch 1
bad = []
for s in samples:
    try:
        with Image.open(s) as im:
            im.convert("RGB").load()
            sz = im.size
        print(f"  decoded {s.parent.name}/{s.name}  {sz[0]}x{sz[1]}")
    except Exception as e:
        bad.append(f"{s}: {type(e).__name__}")
require("data: sample images decode", not bad, "; ".join(bad) if bad else f"{len(samples)} ok")

## 4 · Test knobs

Deliberately small — this is a plumbing test, not a pretraining run.

- **`vit_tiny` / patch 16 / 224 px** → a 14×14 = 196-patch grid, ~5.7 M encoder params.
- **`use_bfloat16: false`** → `scaler` is `None` and the plain `loss.backward()` branch runs.
  Upstream pairs `use_bfloat16` with a **fp16** `GradScaler` (`src/helper.py:155`), a
  mismatch that is a no-op at best; disabling it also keeps the test valid on a T4
  (sm_75, no bf16).
- **`warmup: 1`** — the shipped configs use `warmup: 40` *epochs*, meaningless at 5 total.
- **`num_workers`** is written into the config but `src/datasets/imagenet1k.py:67` forces it
  to `0` whenever `world_size <= 1`, so dataloading here is single-process regardless. Expect
  it to be the wall-clock bottleneck; that is the known cost of the current fix, not a defect
  in this test.

In [ ]:
#@title 4 · Knobs + config writer
import yaml
from pathlib import Path

MODEL_NAME   = "vit_tiny"   #@param ["vit_tiny", "vit_small", "vit_base", "vit_large", "vit_huge"]
PATCH_SIZE   = 16           #@param {type:"integer"}
CROP_SIZE    = 224          #@param {type:"integer"}
BATCH_SIZE   = 32           #@param {type:"integer"}
NUM_WORKERS  = 4            #@param {type:"integer"}
USE_BFLOAT16 = False        #@param {type:"boolean"}
EPOCHS_A     = 5            #@param {type:"integer"}
EPOCHS_B     = 6            #@param {type:"integer"}
PRED_DEPTH   = 6
PRED_EMB_DIM = 192

checks_reset("cfg:")
CFG_DIR = Path("/content/configs_smoke"); CFG_DIR.mkdir(exist_ok=True)
RUN_DIR = Path("/content/runs/smoke")     # -> config logging.folder
TAG     = "jepa"                          # -> config logging.write_tag
CFG_A, CFG_B = CFG_DIR / "smoke_run_a.yaml", CFG_DIR / "smoke_run_b.yaml"
CKPT    = RUN_DIR / f"{TAG}-latest.pth.tar"
CSV     = RUN_DIR / f"{TAG}_r0.csv"
LOG_A, LOG_B = Path("/content/run_a.log"), Path("/content/run_b.log")

def make_config(path, epochs, load_checkpoint, read_checkpoint=None):
    cfg = {
        "meta": {
            "model_name": MODEL_NAME,
            "pred_depth": PRED_DEPTH,
            "pred_emb_dim": PRED_EMB_DIM,
            "use_bfloat16": USE_BFLOAT16,
            "copy_data": False,
            "load_checkpoint": load_checkpoint,
            "read_checkpoint": read_checkpoint,   # None -> falls back to <tag>-latest.pth.tar
        },
        "data": {
            "root_path": str(DATA_ROOT),
            "image_folder": IMAGE_FOLDER,
            "batch_size": BATCH_SIZE,
            "num_workers": NUM_WORKERS,
            "pin_mem": True,
            "crop_size": CROP_SIZE,
            "crop_scale": [0.3, 1.0],
            "use_gaussian_blur": False,
            "use_horizontal_flip": True,
            "use_color_distortion": False,
            "color_jitter_strength": 0.0,
        },
        "mask": {
            "patch_size": PATCH_SIZE,
            "allow_overlap": False,
            "aspect_ratio": [0.75, 1.5],
            "enc_mask_scale": [0.85, 1.0],
            "pred_mask_scale": [0.15, 0.2],
            "num_enc_masks": 1,
            "num_pred_masks": 4,
            "min_keep": 10,
        },
        "optimization": {
            "epochs": epochs,
            "warmup": 1,
            "ema": [0.996, 1.0],
            "ipe_scale": 1.0,
            "lr": 0.001,
            "start_lr": 0.0002,
            "final_lr": 1.0e-06,
            "weight_decay": 0.04,
            "final_weight_decay": 0.4,
        },
        "logging": {"folder": str(RUN_DIR) + "/", "write_tag": TAG},
    }
    Path(path).write_text(yaml.dump(cfg, sort_keys=True), encoding="utf-8")
    return cfg

GRID = CROP_SIZE // PATCH_SIZE
IPE  = N_IMAGES // BATCH_SIZE          # DataLoader is built with drop_last=True
print(f"model      : {MODEL_NAME}/{PATCH_SIZE} @ {CROP_SIZE}px -> {GRID}x{GRID} = {GRID*GRID} patches")
print(f"data       : {N_IMAGES} images / batch {BATCH_SIZE} -> {IPE} iters per epoch")
print(f"run A      : {EPOCHS_A} epochs = {IPE*EPOCHS_A} iters")
print(f"run B      : epoch {EPOCHS_B} only = {IPE} iters (after {IPE*EPOCHS_A} fast-forward steps)")
print(f"run dir    : {RUN_DIR}")
print(f"csv / ckpt : {CSV.name} / {CKPT.name}\n")

require("cfg: iters-per-epoch > 0", IPE > 0,
        f"{N_IMAGES} images < batch_size {BATCH_SIZE}; lower BATCH_SIZE")
require("cfg: run B extends run A by 1 epoch", EPOCHS_B == EPOCHS_A + 1,
        f"EPOCHS_A={EPOCHS_A}, EPOCHS_B={EPOCHS_B}")
check("cfg: warmup fits inside run A", 1 < EPOCHS_A, f"warmup=1 epoch of {EPOCHS_A}")

## 5 · Launcher

Runs `main.py` as a child process so each run gets a clean interpreter — the same thing that
happens on the cluster, and the only way `load_checkpoint` is exercised for real. stdout is
streamed live into the notebook and teed to a log file the later cells parse.

`init_model` logs the entire encoder `repr` (~200 lines for a ViT); that block is collapsed
in the notebook view and kept in full in the log file.

In [ ]:
#@title 5 · Training launcher (streams stdout, tees to a log file)
import subprocess, sys, time, os, signal
from pathlib import Path

def run_training(cfg_path, log_path, hide_model_repr=True, tail_on_error=25):
    # launch main.py as a child process; returns (returncode, elapsed_seconds)
    env = dict(os.environ, PYTHONUNBUFFERED="1")
    cmd = [sys.executable, "-u", "main.py", "--fname", str(cfg_path), "--devices", "cuda:0"]
    print(f"$ {' '.join(cmd)}\n  cwd = {REPO}\n" + "-" * 82)
    t0, in_repr, tail = time.time(), False, []
    p = subprocess.Popen(cmd, cwd=str(REPO), env=env, text=True, bufsize=1,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    try:
        with open(log_path, "w", encoding="utf-8") as lf:
            for line in p.stdout:
                lf.write(line)                    # log file always gets everything
                tail.append(line)
                if len(tail) > tail_on_error:
                    tail.pop(0)
                if hide_model_repr:
                    if line.startswith("VisionTransformer("):
                        in_repr = True
                        print("VisionTransformer(...)      [repr collapsed -- full text in the log file]")
                        continue
                    if in_repr:
                        if line.rstrip("\n") == ")":
                            in_repr = False
                        continue
                print(line, end="")
            p.wait()
    except KeyboardInterrupt:
        p.terminate(); p.wait(timeout=30)
        print("\n!! interrupted -- child terminated")
        raise
    dt = time.time() - t0
    print("-" * 82)
    print(f"exit={p.returncode}   elapsed={dt/60:.2f} min   log={log_path}")
    if p.returncode != 0:
        print(f"\nlast {len(tail)} lines before the failure:")
        for l in tail:
            print("   " + l.rstrip())
    return p.returncode, dt

print("launcher ready")

## 6 · Run A — 5 epochs from scratch

`load_checkpoint: false`. `src/train.py:378` calls `save_checkpoint(epoch+1)` after **every**
epoch, so `jepa-latest.pth.tar` is rewritten five times; the numbered `jepa-ep{N}.pth.tar`
snapshots only appear every `checkpoint_freq = 50` epochs, so a 5-epoch run produces the
`latest` file and nothing else. That is what run B resumes from.

The run directory is wiped first so the CSV and checkpoint examined below are unambiguously
from this test.

In [ ]:
#@title 6 · RUN A -- 5 epochs from scratch
import shutil
checks_reset("runA:")

shutil.rmtree(RUN_DIR, ignore_errors=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)   # train.py dumps params-ijepa.yaml here WITHOUT mkdir

make_config(CFG_A, epochs=EPOCHS_A, load_checkpoint=False)
print(CFG_A.read_text(), end="\n")

rc_a, dt_a = run_training(CFG_A, LOG_A)
log_a = LOG_A.read_text(encoding="utf-8", errors="replace")

import re
require("runA: exited cleanly", rc_a == 0, f"exit code {rc_a}")
check("runA: ran as a single rank", "Initialized (rank/world-size) 0/1" in log_a,
      "world_size==1 -> DDP wrap and NCCL both skipped")
check("runA: no process-group init", "init_process_group" not in log_a)
check("runA: params-ijepa.yaml dumped", (RUN_DIR / "params-ijepa.yaml").is_file())
check("runA: checkpoint written", CKPT.is_file(),
      f"{CKPT.stat().st_size/1024**2:.1f} MiB" if CKPT.is_file() else "missing")
# log lines are emitted as 'INFO:root:Epoch N' -- match the tail, not the line start
eps_a = re.findall(r"Epoch (\d+)\s*$", log_a, flags=re.M)
check("runA: ran epochs 1..%d" % EPOCHS_A, eps_a == [str(i) for i in range(1, EPOCHS_A + 1)],
      f"epochs seen: {eps_a}")
check("runA: no nan loss", "loss is nan" not in log_a)
print(f"\nrun A: {dt_a/60:.2f} min total, {dt_a/60/max(EPOCHS_A,1):.2f} min/epoch")
print("note: 'SLURM vars not set (distributed training not available)' in the log above is")
print("      expected off-cluster -- it is train.py's second init_distributed() probe, and")
print("      it is exactly the branch that now returns (1, 0) instead of hanging on NCCL.")

## 7 · Checkpoint contents + `_match_state_dict_prefix` round-trip

Two independent things are checked.

**(a) What run A actually wrote.** With `world_size == 1`, `src/train.py:222` skips the
`DistributedDataParallel` wrap, so the saved keys are *bare* — no `module.` prefix — and
`epoch` reads `5`.

**(b) The prefix-normalisation fix.** A checkpoint saved by a multi-GPU run carries `module.`
on every key. Feeding that to a bare model raises, and `load_checkpoint` swallows the
exception at `src/helper.py:71`, quietly restarting from epoch 0 — a silent, expensive
failure. This cell builds a synthetic `module.`-prefixed dict, confirms the raw load *does*
fail, then confirms `_match_state_dict_prefix` makes it load with zero missing/unexpected
keys. The bare→bare and bare→wrapped directions are checked too, so the helper can't be
"passing" by rewriting everything unconditionally.

In [ ]:
#@title 7 · Checkpoint contents + prefix-normalisation round-trip
import torch, sys, logging, gc
from pathlib import Path
checks_reset("ckpt:")

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from src.helper import _match_state_dict_prefix, init_model

require("ckpt: file exists", CKPT.is_file(), str(CKPT))
ck = torch.load(CKPT, map_location="cpu", weights_only=True)

print(f"file       : {CKPT}  ({CKPT.stat().st_size/1024**2:.2f} MiB)")
print(f"top-level  : {sorted(ck.keys())}")
print(f"epoch      : {ck['epoch']}")
print(f"world_size : {ck['world_size']}")
print(f"batch_size : {ck['batch_size']}")
print(f"lr         : {ck['lr']}")
print(f"loss       : {ck['loss']:.5f}   (run A last-epoch average)")
print(f"scaler     : {ck['scaler']}   (None because use_bfloat16={USE_BFLOAT16})")

enc_keys = list(ck["encoder"].keys())
n_pref = sum(k.startswith("module.") for k in enc_keys)
print(f"\nencoder tensors: {len(enc_keys)}, with 'module.' prefix: {n_pref}")
print(f"first three    : {enc_keys[:3]}\n")

check("ckpt: epoch == %d" % EPOCHS_A, ck["epoch"] == EPOCHS_A, f"got {ck['epoch']}")
check("ckpt: world_size == 1", ck["world_size"] == 1, f"got {ck['world_size']}")
check("ckpt: no module. prefix (DDP wrap skipped)", n_pref == 0,
      f"{n_pref} of {len(enc_keys)} keys prefixed")
check("ckpt: scaler is None for fp32 path", ck["scaler"] is None or USE_BFLOAT16)

# ---- (b) prefix normalisation, all three directions ------------------------------
_lvl = logging.getLogger().level
logging.getLogger().setLevel(logging.WARNING)      # init_model logs the whole encoder repr
enc, _pred = init_model(device=torch.device("cpu"), patch_size=PATCH_SIZE,
                        crop_size=CROP_SIZE, pred_depth=PRED_DEPTH,
                        pred_emb_dim=PRED_EMB_DIM, model_name=MODEL_NAME)
logging.getLogger().setLevel(_lvl)

ddp_style = {f"module.{k}": v for k, v in ck["encoder"].items()}

raw_failed = False
try:
    enc.load_state_dict(ddp_style)
except Exception as e:
    raw_failed = True
    print(f"raw DDP-style dict rejected, as expected: {type(e).__name__}")
check("ckpt: raw DDP dict rejected without the fix", raw_failed,
      "if this passes, the load was not actually strict")

fixed = _match_state_dict_prefix(enc, ddp_style)
msg = enc.load_state_dict(fixed)
print(f"after _match_state_dict_prefix -> {msg}")
check("ckpt: module.-prefixed dict loads into a bare model",
      not msg.missing_keys and not msg.unexpected_keys,
      f"missing={len(msg.missing_keys)}, unexpected={len(msg.unexpected_keys)}")

same = _match_state_dict_prefix(enc, ck["encoder"])
check("ckpt: bare -> bare is a no-op", list(same.keys()) == list(ck["encoder"].keys()),
      "helper must not rewrite keys that already match")

wrapped = torch.nn.Module(); wrapped.module = enc      # stand-in for a DDP-wrapped model
back = _match_state_dict_prefix(wrapped, ck["encoder"])
msg2 = wrapped.load_state_dict(back)
check("ckpt: bare dict loads into a wrapped model",
      all(k.startswith("module.") for k in back)
      and not msg2.missing_keys and not msg2.unexpected_keys,
      "reverse direction (single-GPU ckpt -> multi-GPU resume)")

del enc, _pred, ddp_style, fixed, same, wrapped, back, ck
gc.collect()

## 8 · Run B — resume and run the 6th epoch

`load_checkpoint: true` with `read_checkpoint: null`, so `load_path` falls back to
`jepa-latest.pth.tar` (`src/train.py:150`). `load_checkpoint()` returns `epoch = 5`, the loop
becomes `range(5, 6)` — exactly one more epoch — and the LR schedule, WD schedule, momentum
generator and mask-collator counter are each fast-forwarded `start_epoch * ipe` steps first
(`src/train.py:245-249`).

**A real caveat, stated rather than hidden:** raising `epochs` 5 → 6 also moves the cosine
horizon, since `T_max = ipe_scale * num_epochs * ipe` (`src/helper.py:149`). Run B's LR at
step *n* is therefore *not* the LR run A would have had at step *n*. That is inherent to
*extending* a run, not a defect in the resume — for a fidelity-preserving resume (after a
preemption, say) keep `epochs` fixed and simply relaunch. Cell 9 checks what actually
matters: that the LR resumed **mid-schedule** rather than restarting at the top of warmup.

In [ ]:
#@title 8 · RUN B -- resume from latest, run epoch 6 only
checks_reset("runB:")
ck_mtime_before = CKPT.stat().st_mtime

make_config(CFG_B, epochs=EPOCHS_B, load_checkpoint=True, read_checkpoint=None)
print(CFG_B.read_text(), end="\n")

rc_b, dt_b = run_training(CFG_B, LOG_B)
require("runB: exited cleanly", rc_b == 0, f"exit code {rc_b}")
print(f"\nrun B: {dt_b/60:.2f} min for 1 resumed epoch")

## 9 · Did it actually resume?

Three ways a "successful" resume can be a lie, all checked here:

1. `load_checkpoint` threw and got swallowed → the log carries
   `Encountered exception when loading checkpoint` and training silently restarted from
   random weights.
2. The checkpoint loaded but `start_epoch` was wrong → run B would re-run epochs it already
   did. It must print `Epoch 6` and nothing else.
3. The weights loaded but the **schedulers** did not fast-forward (`src/train.py:245`) → run
   B's first logged LR would come out *identical* to run A's first LR, because both would be
   warmup step 1. Note the test is that equality, not a fixed threshold: with `warmup: 1` the
   LR *rises* to `lr` during epoch 1 and is already well **below** `start_lr` by epoch 5, so
   "is it above 2e-4" would be exactly backwards.

In [ ]:
#@title 9 · Verify the resume actually happened
import re, torch
checks_reset("resume:")

log_a = LOG_A.read_text(encoding="utf-8", errors="replace")
log_b = LOG_B.read_text(encoding="utf-8", errors="replace")

loaded  = re.findall(r"loaded pretrained encoder from epoch (\d+)", log_b)
eps_b   = re.findall(r"Epoch (\d+)\s*$", log_b, flags=re.M)   # lines read 'INFO:root:Epoch N'
excs    = re.findall(r"Encountered exception when loading checkpoint (.+)", log_b)
lr_of   = lambda s: [float(x) for x in re.findall(r"\[lr: ([0-9.e+-]+)\]", s)]
lrs_a, lrs_b = lr_of(log_a), lr_of(log_b)

print(f"'loaded pretrained ... epoch N' lines : {loaded}")
print(f"epochs executed by run B             : {eps_b}")
print(f"checkpoint-load exceptions           : {excs if excs else 'none'}")
print(f"run A  first / peak / last lr        : {lrs_a[0]:.3e} / {max(lrs_a):.3e} / {lrs_a[-1]:.3e}")
print(f"run B  first / last lr               : {lrs_b[0]:.3e} / {lrs_b[-1]:.3e}\n")

require("resume: no swallowed load exception", not excs,
        excs[0] if excs else "load_checkpoint completed")
check("resume: all 3 modules loaded from epoch %d" % EPOCHS_A,
      loaded == [str(EPOCHS_A)] * 3,
      f"encoder/predictor/target_encoder -> {loaded}")
check("resume: run B executed epoch %d only" % EPOCHS_B, eps_b == [str(EPOCHS_B)],
      f"epochs seen: {eps_b}")
# Without the src/train.py:245 fast-forward loop, run B's very first scheduler.step()
# would land on warmup step 1 and return *exactly* run A's first lr. It doesn't -- it
# picks up on the far side of the cosine. That equality is the sharp test; an absolute
# threshold is not, because by epoch 5 the cosine has already decayed below start_lr.
ff_ok = bool(lrs_b) and abs(lrs_b[0] - lrs_a[0]) > 1e-12 and lrs_b[0] < lrs_a[0]
check("resume: schedulers fast-forwarded", ff_ok,
      f"run B opened at lr {lrs_b[0]:.3e}, not the from-scratch {lrs_a[0]:.3e}" if ff_ok
      else f"run B opened at lr {lrs_b[0]:.3e} == run A's first lr -- schedule restarted")

ck2 = torch.load(CKPT, map_location="cpu", weights_only=True)
print(f"checkpoint epoch after run B : {ck2['epoch']}  (was {EPOCHS_A})")
check("resume: checkpoint advanced to epoch %d" % EPOCHS_B, ck2["epoch"] == EPOCHS_B,
      f"got {ck2['epoch']}")
check("resume: checkpoint file rewritten", CKPT.stat().st_mtime > ck_mtime_before)
del ck2

## 10 · Results

`CSVLogger` opens with mode `'+a'` (`src/utils/logging.py:37`), so run B **appends** to run
A's `jepa_r0.csv` and re-emits the header row at the boundary. That repeated header is the
run separator used below — and is itself independent evidence that two separate processes
wrote this one file.

In [ ]:
#@title 10 · Per-epoch table + loss curve
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
checks_reset("results:")

raw = CSV.read_text(encoding="utf-8").splitlines()
runs, cur = [], None
for line in raw:
    if line.startswith("epoch,"):            # a new process began writing
        cur = []; runs.append(cur); continue
    if not line.strip():
        continue
    e, i, loss, ma, mb, ms = line.split(",")
    cur.append(dict(run=len(runs) - 1, epoch=int(e), itr=int(i), loss=float(loss),
                    maskA=float(ma), maskB=float(mb), ms=float(ms)))

rows  = [r for blk in runs for r in blk]
split = len(runs[0]) if runs else 0
print(f"{CSV}")
print(f"{len(runs)} run block(s), {len(rows)} logged iterations, split at {split}\n")
require("results: two run blocks in the CSV", len(runs) == 2,
        f"expected 2 header rows (one per process), found {len(runs)}")

print(f"{'epoch':>5} {'run':>4} {'iters':>6} {'mean loss':>10} {'ctx patch':>10} "
      f"{'tgt patch':>10} {'ms/itr':>8}")
print("-" * 60)
per_epoch = {}
for ep in sorted({r['epoch'] for r in rows}):
    sel = [r for r in rows if r["epoch"] == ep]
    per_epoch[ep] = float(np.mean([r["loss"] for r in sel]))
    print(f"{ep:>5} {'AB'[sel[0]['run']]:>4} {len(sel):>6} {per_epoch[ep]:>10.5f} "
          f"{np.mean([r['maskA'] for r in sel]):>10.1f} "
          f"{np.mean([r['maskB'] for r in sel]):>10.1f} "
          f"{np.mean([r['ms'] for r in sel]):>8.1f}")

print(f"\nrun A : {dt_a/60:6.2f} min   {EPOCHS_A} epochs   ({dt_a/60/EPOCHS_A:.2f} min/epoch)")
print(f"run B : {dt_b/60:6.2f} min   1 resumed epoch")

check("results: every epoch 1..%d logged" % EPOCHS_B,
      sorted(per_epoch) == list(range(1, EPOCHS_B + 1)), f"epochs {sorted(per_epoch)}")
check("results: loss decreased over the run",
      per_epoch[EPOCHS_B] < per_epoch[1],
      f"epoch 1 {per_epoch[1]:.4f} -> epoch {EPOCHS_B} {per_epoch[EPOCHS_B]:.4f}")
check("results: no nan/inf loss", np.isfinite([r["loss"] for r in rows]).all())

# ---- loss curve ------------------------------------------------------------------
C_A, C_B, INK, MUTED, GRIDC = "#2a78d6", "#eb6834", "#1f1f1c", "#8a8a80", "#e4e4de"
loss = np.array([r["loss"] for r in rows]); step = np.arange(len(rows))
W = max(3, min(25, len(rows) // 20))
smooth = np.convolve(loss, np.ones(W) / W, mode="valid")
sm_x = step[W - 1:]; in_a = sm_x < split

fig, ax = plt.subplots(figsize=(9.5, 4.2), dpi=130)
ax.plot(step[:split], loss[:split], lw=1, color=C_A, alpha=0.28)
ax.plot(step[split:], loss[split:], lw=1, color=C_B, alpha=0.28)
ax.plot(sm_x[in_a],  smooth[in_a],  lw=2, color=C_A, label=f"run A · epochs 1–{EPOCHS_A}")
ax.plot(sm_x[~in_a], smooth[~in_a], lw=2, color=C_B, label=f"run B · epoch {EPOCHS_B} (resumed)")

ax.axvline(split, color=MUTED, lw=1, ls=(0, (4, 3)))
ax.annotate(f"resume\n(ckpt epoch {EPOCHS_A})", xy=(split, ax.get_ylim()[1]),
            xytext=(-7, -4), textcoords="offset points", va="top", ha="right",
            fontsize=8.5, color=MUTED)

ax.set_title(f"I-JEPA smoke test · {MODEL_NAME}/{PATCH_SIZE} · smooth-L1 loss per iteration",
             fontsize=11, color=INK, loc="left", pad=10)
ax.set_xlabel("logged iteration (global)", fontsize=9, color=MUTED)
ax.set_ylabel("loss", fontsize=9, color=MUTED)
ax.grid(axis="y", lw=0.6, color=GRIDC); ax.set_axisbelow(True)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
for s in ("left", "bottom"):
    ax.spines[s].set_color("#d5d5cd")
ax.tick_params(colors=MUTED, labelsize=8.5)
ax.legend(frameon=False, fontsize=9, labelcolor=INK, loc="lower left")
fig.tight_layout(); plt.show()

print(f"faint line = raw per-iteration loss; solid = {W}-iteration moving average")

## 11 · Verdict

In [ ]:
#@title 11 · Verdict + artifacts
from pathlib import Path

groups, order = {}, []
for c in CHECKS:
    g = c["name"].split(":", 1)[0]
    if g not in groups:
        groups[g] = []; order.append(g)
    groups[g].append(c)

LABEL = {"env": "runtime", "repo": "source fingerprint", "data": "dataset",
         "cfg": "config", "runA": "run A (5 epochs)", "ckpt": "checkpoint + prefix fix",
         "runB": "run B (resume)", "resume": "resume verification", "results": "results"}

n_ok = sum(c["ok"] for c in CHECKS)
print(f"{'':2}{'check':<52}{'result':>8}")
print("=" * 62)
for g in order:
    print(f"\n-- {LABEL.get(g, g)} " + "-" * max(0, 56 - len(LABEL.get(g, g))))
    for c in groups[g]:
        name = c["name"].split(":", 1)[1].strip()
        print(f"  {name:<52}{'PASS' if c['ok'] else 'FAIL':>8}")
        if not c["ok"] and c["detail"]:
            print(f"    -> {c['detail']}")
print("=" * 62)
print(f"{n_ok}/{len(CHECKS)} checks passed")
print("\n" + ("VERDICT: PASS -- single-GPU training and resume both behave correctly."
              if n_ok == len(CHECKS) else
              f"VERDICT: FAIL -- {len(CHECKS)-n_ok} check(s) failed; see the -> lines above."))

print(f"\n\n{'artifact':<50}{'size':>10}")
print("-" * 60)
for p in sorted(RUN_DIR.rglob("*")):
    if p.is_file():
        print(f"{str(p.relative_to(RUN_DIR)):<50}{p.stat().st_size/1024**2:>9.2f}M")
for p in (LOG_A, LOG_B):
    if p.is_file():
        print(f"{p.name:<50}{p.stat().st_size/1024**2:>9.2f}M")

print("\nKeep the run:")
print(f"  !cp -r {RUN_DIR} /content/drive/MyDrive/")